In [12]:
API_TOKEN = "CWB-B1034BF8-0855-45E6-8F10-7C7D4DB194AA" 

In [13]:
# =====================================================================
# CWA TWD67 O-A0059-001 
# =====================================================================
ORIGIN_LON = 115.0     
ORIGIN_LAT = 18.0  
RESOLUTION = 0.0125    
GRID_NX = 921          
GRID_NY = 881          
# =====================================================================
ROI_LAT_MIN = 24.75   # South 
ROI_LAT_MAX = 25.30  # North 
ROI_LON_MIN = 121.0  # West 
ROI_LON_MAX = 121.8  # East
# =====================================================================

y_start_idx = int(round((ROI_LAT_MIN - ORIGIN_LAT) / RESOLUTION))
y_end_idx   = int(round((ROI_LAT_MAX - ORIGIN_LAT) / RESOLUTION))

x_start_idx = int(round((ROI_LON_MIN - ORIGIN_LON) / RESOLUTION))
x_end_idx   = int(round((ROI_LON_MAX - ORIGIN_LON) / RESOLUTION))

# =====================================================================
slice_height = y_end_idx - y_start_idx
slice_width  = x_end_idx - x_start_idx
total_grids  = slice_height * slice_width
area_km2 = total_grids * (RESOLUTION * 100)**2 

print("=" * 60)
print("QPESUMS 921x881 網格經緯度轉換")
print("=" * 60)
print(f"{ROI_LAT_MIN}°N ~ {ROI_LAT_MAX}°N")
print(f"{ROI_LON_MIN}°E ~ {ROI_LON_MAX}°E")
print("-" * 60)
print("BASIN INDEX")
print(f"TAIPEI_Y_START, TAIPEI_Y_END = {y_start_idx}, {y_end_idx}")
print(f"TAIPEI_X_START, TAIPEI_X_END = {x_start_idx}, {x_end_idx}")
print(f"總網格數 {total_grids} 格")
print(f"總面積 約 {area_km2:.2f} 平方公里")
print(f"強對流觸發面積門檻 (>10 km²) 約需符合：{(10 / 1.5625):.2f} 格 -> {(10 / 1.5625):.0f} 格")
print("=" * 60)

# =====================================================================
if y_start_idx < 0 or y_end_idx > GRID_NY or x_start_idx < 0 or x_end_idx > GRID_NX:
    print("超出網格邊界")

QPESUMS 921x881 網格經緯度轉換
24.75°N ~ 25.3°N
121.0°E ~ 121.8°E
------------------------------------------------------------
BASIN INDEX
TAIPEI_Y_START, TAIPEI_Y_END = 540, 584
TAIPEI_X_START, TAIPEI_X_END = 480, 544
總網格數 2816 格
總面積 約 4400.00 平方公里
強對流觸發面積門檻 (>10 km²) 約需符合：6.40 格 -> 6 格


In [16]:
import subprocess
import requests
import xml.etree.ElementTree as ET
import numpy as np
from datetime import datetime, timedelta

# yyyy-mm-dd; None = yesterday
MANUAL_DATE = "2026-06-14"  

TAIPEI_Y_START, TAIPEI_Y_END = 540, 584
TAIPEI_X_START, TAIPEI_X_END = 480, 544

DBZ_THRESHOLD = 40.0       # 40 dBZ
GRID_COUNT_THRESHOLD = 6   # 6格

# ==========================================
if MANUAL_DATE is None:
    target_date_obj = datetime.now() - timedelta(days=1)
    print("未手動指定日期，昨天" + target_date_obj.strftime("%Y-%m-%d"))
else:
    try:
        target_date_obj = datetime.strptime(MANUAL_DATE, "%Y-%m-%d")
        print(f"指定日期：{MANUAL_DATE}")
    except ValueError:
        print("日期格式錯誤: 'YYYY-MM-DD'" )
        exit()

TARGET_YEAR  = target_date_obj.strftime("%Y")
TARGET_MONTH = target_date_obj.strftime("%m")
TARGET_DAY   = target_date_obj.strftime("%d")
print("-" * 80)

# ==========================================
# Afternoon
start_dt = datetime(int(TARGET_YEAR), int(TARGET_MONTH), int(TARGET_DAY), 12, 0, 0)
end_dt = datetime(int(TARGET_YEAR), int(TARGET_MONTH), int(TARGET_DAY), 18, 0, 0)

current_dt = start_dt
time_series_results = []
nx, ny = 921, 881

print(f"{'觀測時間 (LST)':<25}{'盆地內≧40dBZ格點數':<18}{'換算面積 (km²)':<15}{'單一時段判定':<10}")
print("-" * 70)

# ==========================================
while current_dt <= end_dt:
    y = current_dt.strftime("%Y")
    m = current_dt.strftime("%m")
    d = current_dt.strftime("%d")
    hh = current_dt.strftime("%H")
    mm = current_dt.strftime("%M")
    ss = current_dt.strftime("%S")
    
    # History API; XML
    url = f"https://opendata.cwa.gov.tw/historyapi/v1/getData/O-A0059-001/{y}/{m}/{d}/{hh}/{mm}/{ss}?Authorization={API_TOKEN}&format=XML"
    
    try:
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            # ➔ 步驟 2-1: 讀取純 XML 位元組內容
            xml_content = response.content
            root = ET.fromstring(xml_content)
            
            # ➔ 步驟 2-2: 全自動搜索隱藏的格點大字串
            raw_text = ""
            for node in root.iter():
                if node.text and node.text.count(',') > 100:
                    raw_text = node.text.strip()
                    break
            
            if raw_text:
                # ➔ 步驟 2-3: NumPy 解碼與重塑
                flat_data = np.fromstring(raw_text, sep=',')[:(nx * ny)]
                full_matrix = flat_data.reshape((ny, nx))
                
                # ➔ 步驟 2-4: 台北盆地切片與物理統計
                taipei_basin = full_matrix[TAIPEI_Y_START:TAIPEI_Y_END, TAIPEI_X_START:TAIPEI_X_END]
                valid_taipei = taipei_basin[taipei_basin >= 0]
                
                if len(valid_taipei) > 0:
                    strong_grids = valid_taipei[valid_taipei >= DBZ_THRESHOLD]
                    grid_count = len(strong_grids)
                    calculated_area = grid_count * 1.5625
                else:
                    grid_count, calculated_area = 0, 0.0
                
                # ➔ 步驟 2-5: 記錄與輸出日誌
                is_single_met = grid_count >= GRID_COUNT_THRESHOLD
                time_series_results.append(is_single_met)
                
                time_label = current_dt.strftime('%Y-%m-%d %H:%M')
                status_text = "🔴 達標" if is_single_met else "⚪ 正常"
                print(f"{time_label:<25}{grid_count:<22}{calculated_area:<18.2f}{status_text:<10}")
                
                current_dt += timedelta(minutes=10)
                continue

        # 晴天、無回波或該時段維護防呆
        time_series_results.append(False)
        time_label = current_dt.strftime('%Y-%m-%d %H:%M')
        print(f"{time_label:<25}{0:<22}{0.00:<18.2f}{'🔵 無回波':<10}")
        
    except Exception as e:
        # 防範個別時段檔案受損
        time_series_results.append(False)
        time_label = current_dt.strftime('%Y-%m-%d %H:%M')
        print(f"{time_label:<25}{0:<22}{0.00:<18.2f}{'❌ 異常跳過':<10}")
        
    current_dt += timedelta(minutes=10)

# ==========================================
# 時間持續性 (連續滿 30 分鐘)
# ==========================================
is_triggered = False
consecutive_counter = 0

for result in time_series_results:
    if result:
        consecutive_counter += 1
        if consecutive_counter >= 3:
            is_triggered = True
            break
    else:
        consecutive_counter = 0

# ==========================================
# Last CHECK
# ==========================================
print("-" * 70)
if is_triggered:
    print(f"\n{TARGET_YEAR}-{TARGET_MONTH}-{TARGET_DAY} 成功觸發午後雷雨個案")
    print("➔ 物理指標：完全滿足 40 dBZ + 10 km² + 30 Mins")
    print("=" * 70 + "\n")
    print("開始自動儲存資料...")

    next_script = "pillow_cropper.py" 
    
    print(f"執行 {next_script}...")
    result = subprocess.run(
        ["python", next_script, TARGET_YEAR, TARGET_MONTH, TARGET_DAY], 
        capture_output=True, 
        text=True
    )
    
    if result.returncode == 0:
        print("DONE! ")
        print(result.stdout) 
    else:
        print("錯誤：")
        print(result.stderr) 
else:
    print(f"\n🟢 {TARGET_YEAR}-{TARGET_MONTH}-{TARGET_DAY} 未達標準。")

指定日期：2026-06-14
--------------------------------------------------------------------------------
觀測時間 (LST)               盆地內≧40dBZ格點數      換算面積 (km²)     單一時段判定    
----------------------------------------------------------------------
2026-06-14 12:00         55                    85.94             🔴 達標      
2026-06-14 12:10         92                    143.75            🔴 達標      
2026-06-14 12:20         31                    48.44             🔴 達標      
2026-06-14 12:30         5                     7.81              ⚪ 正常      
2026-06-14 12:40         4                     6.25              ⚪ 正常      
2026-06-14 12:50         3                     4.69              ⚪ 正常      
2026-06-14 13:00         2                     3.12              ⚪ 正常      
2026-06-14 13:10         0                     0.00              ⚪ 正常      
2026-06-14 13:20         0                     0.00              ⚪ 正常      
2026-06-14 13:30         0                     0.00              ⚪ 正常      
202